# Heart Disease Prediction

Predicting whether a person has heart disease based on medical attributes.
Dataset: https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset


## 1. Load the data

Downloaded automatically with kagglehub.


In [ ]:
!pip install kagglehub -q

import kagglehub, os
path = kagglehub.dataset_download('johnsmith88/heart-disease-dataset')
print(os.listdir(path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(os.path.join(path, 'heart.csv'))
df.head()


## 2. Quick look at the data


In [ ]:
print(df.shape)
df.info()
print(df.isnull().sum())
df['target'].value_counts()


In [ ]:
df['target'].value_counts().plot(kind='bar')
plt.title('target distribution')
plt.show()

plt.figure(figsize=(14,8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.show()


## 3. Cleaning

This dataset is already pretty clean, no missing values. Just a quick check for duplicates.


In [ ]:
df = df.drop_duplicates()
df.shape


## 4. Training

Split the data, scale it, then try 3 classifiers and compare them.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

scores = {}
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted[name] = model
    scores[name] = {
        'accuracy': round(accuracy_score(y_test, pred), 4),
        'precision': round(precision_score(y_test, pred), 4),
        'recall': round(recall_score(y_test, pred), 4),
        'f1': round(f1_score(y_test, pred), 4)
    }

pd.DataFrame(scores).T


## 5. Best model and saving

Picking the best F1 score, plotting the confusion matrix, then saving with joblib.


In [ ]:
best_name = pd.DataFrame(scores).T['f1'].idxmax()
best_model = fitted[best_name]
print('best model:', best_name)

pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.show()

print(classification_report(y_test, pred))


In [ ]:
os.makedirs('model_artifacts', exist_ok=True)
joblib.dump(best_model, 'model_artifacts/heart_model.pkl')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')
joblib.dump(X.columns.tolist(), 'model_artifacts/features.pkl')
os.listdir('model_artifacts')
